In [1]:
import os
import ROOT as root
import numpy as np
from array import array
import glob
import re

file = root.TFile("/Users/icosivi/cernbox/MTD/QAQC/root_files/UFSDk2_16x16_IVtree.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
#type = array('i', [0])
row = array('i', [0])
column = array('i', [0])

V = root.std.vector("float")()
IBACK = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
#tree.Branch("type", type,'type/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')


V.reserve(1000)
IBACK.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IBACK", "std::vector<float>", IBACK)

txt_files = glob.glob("/Users/icosivi/cernbox/MTD/QAQC/FBKdata/UFSD_K2/*.txt")

counter = 0

for txt in txt_files:
    #print(txt)
    V.clear()
    #IBACK.clear()

    with open(txt,"r") as t:
        lines_list = t.readlines()
        header_v = lines_list[0]
        
        header_v.strip()
        hv = re.split("\s+", header_v)
        
        for m in hv[8:]:
            if not m.isspace():
                if m:
                    #print(m)
                    V.push_back( abs(float(m)) )

        lines = lines_list[1:]
        
        for line in lines:
            line.strip()
            IBACK.clear()
            ll = re.split( '\s+', line)
            #print(ll[4])
            if ll[5] == "I_BACK[A]":
                event[0] = counter
                wafer[0] = int( ll[0] )
                column[0] = int( ll[1] )
                row[0] = int( ll[2] )
                
                for q in ll[6:]:
                    if not q.isspace():
                        if q:
                            IBACK.push_back( abs(float(q)) )
                            #if counter==2:
                                #print(q)          
                #if not any(x == "PIN" for x in sensor_types):
                tree.Fill()
                counter += 1

tree.Write()
file.Write()
file.Close()

Welcome to JupyROOT 6.30/04
